In [ ]:
!pip install llama-index sentence-transformers transformers torch pypdf streamlit

In [ ]:
!pip install llama-index-embeddings-huggingface llama-index-llms-huggingface


In [ ]:
!pip install bitsandbytes accelerate

In [ ]:
import os

os.makedirs("data/uploads", exist_ok=True)
print("Folder Created!")

Folder Created!


In [ ]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.readers import SimpleDirectoryReader
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# 1. Load documents
reader = SimpleDirectoryReader("data/uploads")
documents = reader.load_data()

print(f"Loaded {len(documents)} document chunks")

Loaded 121 document chunks


In [ ]:
# 2. Build embeddings + index
embed_model = HuggingFaceEmbedding(
    model_name="all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=2048,
    max_new_tokens=200,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
Settings.llm = llm
Settings.embed_model = embed_model

In [ ]:
# 3. Query engine
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print("Rag system ready!")

Rag system ready!


In [ ]:
response = query_engine.query(
    "What is the net profit?"
)

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Based on the provided financial statements, the net income for Apple Inc. in the year ended September 28, 2024, was $93,736 million. This net income is the starting point for calculating net profit. However, it's important to note that net income is not the same as net profit, which is the amount of income that remains after all expenses, including taxes, have been deducted. 
The financial statements provided do not include a line item for net profit. However, we can calculate net profit by subtracting the income taxes expense from net income. Using the data provided, net income was $93,736 million and income taxes expense was $29,749 million. Therefore, net profit for Apple Inc. in the year ended September 28, 2024, was $63,987 million ($93,736 - $2


In [ ]:
# 4. Chat wrapper

chat_history = []

def chat(query):
    global chat_history

    context = ""
    if chat_history:
        last_turn = chat_history[-1]
        context = f"""
Background context (do NOT answer this again):
Previous question: {last_turn['user']}
Previous answer: {last_turn['assistant']}
"""

    full_query = f"""
You are a financial document assistant.

RULES:
- Answer ONLY the user's latest question.
- Do NOT repeat or answer previous questions.
- Use the background context ONLY if needed for comparison.
- Base answers ONLY on the provided documents.

{context}

User's latest question:
{query}

Answer:
"""

    response = query_engine.query(full_query)

    chat_history.append({
        "user": query,
        "assistant": response.response
    })

    return response



In [ ]:
resp = chat("What is the total net sales for 2024?")
print(resp.response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



The total net sales for 2024 can be found in the Consolidated Statements of Operations in Part II, Item 8 of the Form 10-K for the fiscal year ended September 28, 2024. The figure for net sales for the year ended September 28, 2024, is reported there. The context provided does not contain the specific figure for net sales for the year ended September 28, 2024. However, it does mention several new product, service and software offerings that were announced during fiscal year 2024, which could have contributed to the net sales for that year. Therefore, to fully understand the net sales for 2024, it is recommended to refer to the Consolidated Statements of Operations in Part II, Item 8 of the Form 10-K for the fiscal year ended September 28, 20


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr

def ui_chat(user_input, history):
    response = chat(user_input)
    return response.response

with gr.Blocks() as demo:
    gr.Markdown("# 📊 Financial Document Chatbot")
    gr.Markdown("Ask questions based on uploaded financial reports")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask a question about the document...")
    clear = gr.Button("Clear")

    def respond(message, chat_history):
        response = chat(message)
        chat_history.append((message, response.response))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)


/tmp/ipython-input-565264878.py:11: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c06d0c90fb81726a58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
